# Initialization

In [ ]:
!pip install pyspark

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [7]:
spark = SparkSession.builder \
    .appName("GooglePlayStore_ETL") \
    .getOrCreate()

In [8]:
df = spark.read.csv(
    "/content/googleplaystore.csv",
    header=True,
    inferSchema=True
)

In [9]:
df.show(5)

+--------------------+--------------+------+-------+----+-----------+----+-----+--------------+--------------------+----------------+------------------+------------+
|                 App|      Category|Rating|Reviews|Size|   Installs|Type|Price|Content Rating|              Genres|    Last Updated|       Current Ver| Android Ver|
+--------------------+--------------+------+-------+----+-----------+----+-----+--------------+--------------------+----------------+------------------+------------+
|Photo Editor & Ca...|ART_AND_DESIGN|   4.1|    159| 19M|    10,000+|Free|    0|      Everyone|        Art & Design| January 7, 2018|             1.0.0|4.0.3 and up|
| Coloring book moana|ART_AND_DESIGN|   3.9|    967| 14M|   500,000+|Free|    0|      Everyone|Art & Design;Pret...|January 15, 2018|             2.0.0|4.0.3 and up|
|U Launcher Lite –...|ART_AND_DESIGN|   4.7|  87510|8.7M| 5,000,000+|Free|    0|      Everyone|        Art & Design|  August 1, 2018|             1.2.4|4.0.3 and up|
|Ske

In [10]:
df.printSchema()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



# Transform

In [11]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

Rows : 10841
Columns : 13


In [12]:
missing = df.select([
    count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
    for c in df.columns
])

missing.show(truncate=False)

+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|App|Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|Genres|Last Updated|Current Ver|Android Ver|
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|0  |0       |0     |0      |0   |0       |0   |0    |1             |0     |0           |1          |1          |
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+



In [13]:
df.filter(~col("Reviews").rlike("^[0-9]+$")).show(truncate=False)

+---------------------------------------+-------------+-----------+-------------+-------------------+--------+------+------------------+--------------+-----------------+------------+-----------+-----------------+
|App                                    |Category     |Rating     |Reviews      |Size               |Installs|Type  |Price             |Content Rating|Genres           |Last Updated|Current Ver|Android Ver      |
+---------------------------------------+-------------+-----------+-------------+-------------------+--------+------+------------------+--------------+-----------------+------------+-----------+-----------------+
|"Yanosik: ""antyradar""                | traffic jams| navigation| camera"     |MAPS_AND_NAVIGATION|4.4     |102248|Varies with device|5,000,000+    |Free             |0           |Everyone   |Maps & Navigation|
|"Women""s Health Tips(Breast           |Face         |Body       |weight lose)"|HEALTH_AND_FITNESS |4.2     |2509  |5.0M              |1,000,000+  

In [14]:
df = df.filter(col("Reviews").rlike("^[0-9]+$"))

In [15]:
print("Before:", df.count())

df = df.dropDuplicates()

print("After :", df.count())

Before: 10838
After : 10355


### Standardizing column names

In [16]:
df = (df
      .withColumnRenamed("Content Rating", "ContentRating")
      .withColumnRenamed("Last Updated", "LastUpdated")
      .withColumnRenamed("Current Ver", "CurrentVersion")
      .withColumnRenamed("Android Ver", "AndroidVersion"))
df.columns

['App',
 'Category',
 'Rating',
 'Reviews',
 'Size',
 'Installs',
 'Type',
 'Price',
 'ContentRating',
 'Genres',
 'LastUpdated',
 'CurrentVersion',
 'AndroidVersion']

Converting installs to int

In [17]:
df = df.withColumn(
    "Installs",
    regexp_replace(col("Installs"), "[+,]", "")
)

df = df.withColumn(
    "Installs",
    col("Installs").cast("long")
)

Removing $ from price and converting to double

In [18]:
df = df.withColumn(
    "Price",
    regexp_replace(col("Price"), "\\$", "")
)

df = df.withColumn(
    "Price",
    col("Price").cast("double")
)

converting file size to a numeric value and sotring in a new column

In [19]:
df = df.withColumn(
    "Size_MB",
    when(col("Size").endswith("M"),
         regexp_replace(col("Size"), "M", "").cast("double"))
    .when(col("Size").endswith("k"),
         regexp_replace(col("Size"), "k", "").cast("double")/1024)
    .otherwise(None)
)

In [48]:
mean_size = df.select(mean("Size_MB")).first()[0]

df = df.na.fill({"Size_MB": mean_size})

In [49]:
df.show()

+--------------------+------------------+------+-------+------------------+----------+----+-----+-------------+----------------+-----------------+------------------+------------------+------------------+
|                 App|          Category|Rating|Reviews|              Size|  Installs|Type|Price|ContentRating|          Genres|      LastUpdated|    CurrentVersion|    AndroidVersion|           Size_MB|
+--------------------+------------------+------+-------+------------------+----------+----+-----+-------------+----------------+-----------------+------------------+------------------+------------------+
|Free VIN Report f...| AUTO_AND_VEHICLES|   4.6|   2431|              2.4M|    100000|Free|  0.0|     Everyone| Auto & Vehicles|    June 12, 2018|              1.37|        4.0 and up|               2.4|
|Opera Browser: Fa...|     COMMUNICATION|   4.4|2473509|Varies with device| 100000000|Free|  0.0|     Everyone|   Communication|    July 31, 2018|  47.1.2249.129326|Varies with device|

### Creating new columns

In [50]:
df = df.withColumn(
    "Is_Paid",
    when(col("Type")=="Paid",1).otherwise(0)
)

In [51]:
df = df.withColumn(
    "Rating_Category",
    when(col("Rating") >= 4.5, "Excellent")
    .when(col("Rating") >= 4.0, "Good")
    .when(col("Rating") >= 3.0, "Average")
    .otherwise("Poor")
)

In [52]:
df.show(10)

+--------------------+-----------------+------+-------+------------------+---------+----+-----+-------------+---------------+--------------+------------------+------------------+------------------+-------+---------------+
|                 App|         Category|Rating|Reviews|              Size| Installs|Type|Price|ContentRating|         Genres|   LastUpdated|    CurrentVersion|    AndroidVersion|           Size_MB|Is_Paid|Rating_Category|
+--------------------+-----------------+------+-------+------------------+---------+----+-----+-------------+---------------+--------------+------------------+------------------+------------------+-------+---------------+
|Free VIN Report f...|AUTO_AND_VEHICLES|   4.6|   2431|              2.4M|   100000|Free|  0.0|     Everyone|Auto & Vehicles| June 12, 2018|              1.37|        4.0 and up|               2.4|      0|      Excellent|
|Opera Browser: Fa...|    COMMUNICATION|   4.4|2473509|Varies with device|100000000|Free|  0.0|     Everyone|  C

# Loading the completed dataset

In [53]:
df.coalesce(1).write.mode("overwrite").option("header", True).csv("cleaned_google_playstore")